# Assignment 05: Documented Cleaning Pipeline

Work in order: **define → audit → decide → transform → validate → save**. Restart the kernel and run all cells before submission.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd

assert sys.version_info[:3] == (3, 12, 13), 'Use the recorded Python 3.12.13 environment.'
assert np.__version__ == '2.0.2', 'Use NumPy 2.0.2 from requirements.txt.'
assert pd.__version__ == '3.0.5', 'Use pandas 3.0.5 from requirements.txt.'

def _find_assignment_base(start):
    resolved = Path(start).resolve()
    for current in (resolved, *resolved.parents):
        for candidate in (current, current / '05' / 'assignment'):
            manifest_path = candidate / 'data' / 'fixture.json'
            data_path = candidate / 'data' / 'people_raw.csv'
            if manifest_path.is_file() and data_path.is_file():
                return candidate
    raise FileNotFoundError('Could not locate data/fixture.json and data/people_raw.csv.')

BASE_DIR = _find_assignment_base(Path.cwd())
DATA_PATH = BASE_DIR / 'data' / 'people_raw.csv'
MANIFEST_PATH = BASE_DIR / 'data' / 'fixture.json'
OUTPUT_DIR = BASE_DIR / 'output'
AUDIT_PATH = OUTPUT_DIR / 'issue_audit.csv'
CLEANED_PATH = OUTPUT_DIR / 'cleaned_people.csv'
DECISION_LOG_PATH = OUTPUT_DIR / 'decision_log.csv'
EXPECTED_RAW_COLUMNS = ['record_id', 'full_name', 'site', 'status', 'age_text', 'visit_date']
EXACT_DATE_PATTERN = r'[0-9]{4}-[0-9]{2}-[0-9]{2}'
EXPECTED_SOURCE_SHA256 = 'd13dc9676519c81729b33d53ffc2e8fec92e645c6978af7ebf325fcd7147753b'
EXPECTED_MANIFEST = {
    'fixture_id': 'a05-people-cleaning-v1',
    'provenance': 'course-authored synthetic teaching data; no real people',
    'row_meaning': 'one submitted person record',
    'candidate_identifier': ['record_id'],
    'row_count': 12,
    'raw_columns': EXPECTED_RAW_COLUMNS,
    'sha256': EXPECTED_SOURCE_SHA256,
}
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest == EXPECTED_MANIFEST, 'Restore the supplied data/fixture.json manifest.'
source_bytes = DATA_PATH.read_bytes()
assert len(source_bytes) == 570, 'Restore the supplied 570-byte data source.'
assert hashlib.sha256(source_bytes).hexdigest() == EXPECTED_SOURCE_SHA256, 'Restore data/people_raw.csv without editing it.'
print('Verified fixture:', manifest['fixture_id'])

## Task 1 — Define the contract and audit raw data (30 points)

State the contract before changing any values, then implement the complete 15-row audit.

**TODO:** Replace this prompt with a concise contract. State the row meaning and candidate identifier, distinguish raw from clean data, and define **schema**, **sentinel**, **duplicate**, **missing value**, **validation invariant**, and **provenance** before using those terms freely.

In [ ]:
row_meaning = manifest['row_meaning']
candidate_identifier = manifest['candidate_identifier']
raw = pd.read_csv(DATA_PATH, keep_default_na=False)
raw_snapshot = raw.copy(deep=True)

def audit_person_records(raw_table):
    # TODO: return the required 15-row issue,count DataFrame.
    return pd.DataFrame(columns=['issue', 'count'])

issue_audit = audit_person_records(raw)
issue_counts = dict(zip(issue_audit.get('issue', []), issue_audit.get('count', []), strict=True))
assert raw.equals(raw_snapshot), 'Auditing must not mutate raw.'

## Task 2 — Record decisions and transform a copy (40 points)

Create the eight-row decision table first. Then clean a deep copy without filling or inventing values.

In [ ]:
# TODO: replace this empty table with eight ordered, documented decisions.
decision_table = pd.DataFrame(columns=['field', 'issue', 'action', 'reason'])

**TODO:** Explain why forward/backward fill is not justified across these person records, why reviewable missing values remain, and why clean means satisfying a stated contract rather than making data perfect.

In [ ]:
def clean_person_records(raw_table):
    # TODO: implement the documented copy-only transformations.
    return raw_table.copy(deep=True)

cleaned = clean_person_records(raw)
review_queue = cleaned.loc[cleaned['needs_review']].copy() if 'needs_review' in cleaned else cleaned.iloc[0:0].copy()

## Task 3 — Validate, save, and read back (30 points)

Make every invariant explicit and stop before export if any check fails. Then read all three CSV files back with explicit schemas.

In [ ]:
def validate_clean_records(raw_table, raw_before, cleaned_table):
    # TODO: return the complete named boolean validation Series.
    return pd.Series({'implemented': False}, dtype='boolean')

validation_results = validate_clean_records(raw, raw_snapshot, cleaned)
assert bool(validation_results.all()), validation_results[~validation_results].index.tolist()

In [ ]:
# TODO: build decision_log, save all three artifacts, read them back with
# explicit schemas, and compare each round trip exactly.
decision_log = None
round_trip = None
audit_round_trip = None
decision_round_trip = None

## Supplied final verification

Do not edit this cell. A fresh run reaches it only after all invariants and round trips pass.

In [ ]:
expected_issue_counts = {
    'schema mismatch': 0,
    'empty full-name tokens': 1,
    'empty date tokens': 1,
    'age sentinel tokens': 3,
    'status sentinel tokens': 1,
    'age parse failures': 1,
    'numeric but noninteger age values': 1,
    'age values outside 0 through 120': 1,
    'date parse failures': 3,
    'rows in exact duplicate sets': 2,
    'rows with repeated candidate IDs': 2,
    'site values needing format normalization': 4,
    'status values needing format normalization': 3,
    'unexpected site values': 0,
    'unexpected non-sentinel status values': 0,
}
assert issue_counts == expected_issue_counts
assert cleaned['record_id'].tolist() == [f'R{number:03d}' for number in range(1, 12)]
assert len(review_queue) == 7
assert raw.equals(raw_snapshot)
assert bool(validation_results.all())
assert all(path.is_file() for path in (AUDIT_PATH, CLEANED_PATH, DECISION_LOG_PATH))
pd.testing.assert_frame_equal(round_trip, cleaned.reset_index(drop=True))
pd.testing.assert_frame_equal(audit_round_trip, issue_audit.reset_index(drop=True))
pd.testing.assert_frame_equal(decision_round_trip, decision_log.reset_index(drop=True))
print('Assignment 05 fresh-run verification passed')